# Example 1: Aircraft MPC Feasibility and Stability

This notebook recreates the lecture experiment around the **Cessna Citation** longitudinal model and compares:

1. **LQR with saturated input**
2. **MPC with input bounds only**
3. **MPC with amplitude and rate constraints**
4. **MPC with an added pitch-angle state constraint**
5. **Short-horizon MPC with and without a terminal cost**

## Quick Start

Run cells from top to bottom. The main experiment cell reproduces the lecture cases directly, and the last cell provides an interactive scenario viewer.


In [ ]:
# Optional dependency install (uncomment if needed)
# %pip install numpy scipy matplotlib pandas ipywidgets cvxpy


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cvxpy as cp

from scipy.signal import cont2discrete
from scipy.linalg import solve_continuous_are, solve_discrete_are
from IPython.display import display, Markdown

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11


## Model and Lecture Setup

The continuous-time linearized aircraft model is

$$
\dot x =
\begin{bmatrix}
-1.2822 & 0 & 0.98 & 0 \\
0 & 0 & 1 & 0 \\
-5.4293 & 0 & -1.8366 & 0 \\
-128.2 & 128.2 & 0 & 0
\end{bmatrix} x
+
\begin{bmatrix}
-0.3 \\
0 \\
-17 \\
0
\end{bmatrix} u
$$

with states

- $x_1$: angle of attack
- $x_2$: pitch angle
- $x_3$: pitch rate
- $x_4$: altitude

and input $u$ equal to the elevator angle.

We discretize the model with sampling time $T_s = 0.25$ s and use the quadratic stage cost

$$
\ell(x,u)=x^TQx+u^TRu, \qquad Q=I,\; R=10.
$$

**Important modeling detail for the slide experiment**

The lecture story only makes sense if the **real actuator** obeys both amplitude and rate limits, while some controllers do **not** model all of those limits. In particular:

- the **input-only MPC** optimizes with $|u_k|\le 0.262$,
- but the simulated actuator still enforces the physical rate limit,
- which creates the non-convergent oscillatory behavior highlighted in the slides.

For the pitch-constrained example, we solve the QP with `SCS`, which is more numerically reliable here than `OSQP` for the hard state-constrained formulation.


In [ ]:
# Continuous-time aircraft model
A_c = np.array([
    [-1.2822,   0.0,   0.98,   0.0],
    [ 0.0,      0.0,   1.0,    0.0],
    [-5.4293,   0.0,  -1.8366, 0.0],
    [-128.2,  128.2,   0.0,    0.0],
], dtype=float)

B_c = np.array([
    [-0.3],
    [ 0.0],
    [-17.0],
    [ 0.0],
], dtype=float)

C = np.array([
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

state_names = ["angle of attack x1", "pitch angle x2", "pitch rate x3", "altitude x4"]
output_names = ["pitch angle", "altitude"]

Ts = 0.25
Q = np.eye(4)
R = np.array([[10.0]])

u_max = 0.262
du_max_phys = 0.524 * Ts
du_max_tight = 0.349 * Ts
pitch_max = 0.349

Ad, Bd, _, _, _ = cont2discrete((A_c, B_c, np.eye(4), np.zeros((4, 1))), Ts)

P_cont = solve_continuous_are(A_c, B_c, Q, R)
K_cont = np.linalg.solve(R, B_c.T @ P_cont)

P_disc = solve_discrete_are(Ad, Bd, Q, R)
K_disc = np.linalg.solve(R + Bd.T @ P_disc @ Bd, Bd.T @ P_disc @ Ad)

open_loop_poles_c = np.linalg.eigvals(A_c)
open_loop_poles_d = np.linalg.eigvals(Ad)
closed_loop_poles_d = np.linalg.eigvals(Ad - Bd @ K_disc)


def actuator_apply(u_des, u_prev, u_bound=u_max, du_bound=None):
    u_sat = float(np.clip(u_des, -u_bound, u_bound))
    if du_bound is None:
        return u_sat
    delta = np.clip(u_sat - u_prev, -du_bound, du_bound)
    return float(np.clip(u_prev + delta, -u_bound, u_bound))


def simulate_open_loop(x0, steps=40):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    for _ in range(steps):
        u = 0.0
        x = Ad @ x + Bd[:, 0] * u
        xs.append(x.copy())
        us.append(u)
    return {"x": np.array(xs), "u": np.array(us), "label": "Open-loop"}


def simulate_lqr_with_saturation(x0, steps=40, use_continuous_gain=True, du_bound=None):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    u_prev = 0.0

    K = K_cont if use_continuous_gain else K_disc
    controller_name = "continuous-time LQR gain" if use_continuous_gain else "discrete-time LQR gain"

    for _ in range(steps):
        u_cmd = float(-(K @ x).item())
        u = actuator_apply(u_cmd, u_prev, u_bound=u_max, du_bound=du_bound)
        x = Ad @ x + Bd[:, 0] * u
        xs.append(x.copy())
        us.append(u)
        u_prev = u

    return {
        "x": np.array(xs),
        "u": np.array(us),
        "label": f"LQR + saturation ({controller_name})",
        "u_prev_final": u_prev,
    }


def solve_mpc_step(
    x_now,
    u_prev,
    horizon,
    rate_bound_model=None,
    pitch_bound=None,
    terminal_cost=None,
    solver=cp.SCS,
):
    X = cp.Variable((4, horizon + 1))
    U = cp.Variable((1, horizon))

    constraints = [X[:, 0] == x_now]
    cost = 0

    for k in range(horizon):
        cost += cp.quad_form(X[:, k], Q) + cp.quad_form(U[:, k], R)
        constraints += [
            X[:, k + 1] == Ad @ X[:, k] + Bd @ U[:, k],
            cp.abs(U[:, k]) <= u_max,
        ]

        if rate_bound_model is not None:
            previous = u_prev if k == 0 else U[:, k - 1]
            constraints.append(cp.abs(U[:, k] - previous) <= rate_bound_model)

        if pitch_bound is not None:
            constraints.append(cp.abs(X[1, k]) <= pitch_bound)

    if pitch_bound is not None:
        constraints.append(cp.abs(X[1, horizon]) <= pitch_bound)

    terminal_matrix = P_disc if terminal_cost is None else terminal_cost
    cost += cp.quad_form(X[:, horizon], terminal_matrix)

    problem = cp.Problem(cp.Minimize(cost), constraints)
    solve_kwargs = {"warm_start": True, "verbose": False}
    if solver == cp.SCS:
        solve_kwargs["eps"] = 1e-5
    problem.solve(solver=solver, **solve_kwargs)

    if U.value is None:
        return None, problem.status

    return float(U.value[0, 0]), problem.status


def simulate_mpc(
    x0,
    steps=40,
    horizon=10,
    rate_bound_model=None,
    rate_bound_actual=None,
    pitch_bound=None,
    terminal_cost=None,
    solver=cp.SCS,
    label="MPC",
):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    u_cmds = []
    statuses = []
    u_prev = 0.0

    for k in range(steps):
        u_cmd, status = solve_mpc_step(
            x_now=x,
            u_prev=u_prev,
            horizon=horizon,
            rate_bound_model=rate_bound_model,
            pitch_bound=pitch_bound,
            terminal_cost=terminal_cost,
            solver=solver,
        )

        statuses.append(status)
        if u_cmd is None:
            break

        u = actuator_apply(u_cmd, u_prev, u_bound=u_max, du_bound=rate_bound_actual)
        x = Ad @ x + Bd[:, 0] * u

        xs.append(x.copy())
        us.append(u)
        u_cmds.append(u_cmd)
        u_prev = u

    return {
        "x": np.array(xs),
        "u": np.array(us),
        "u_cmd": np.array(u_cmds),
        "statuses": statuses,
        "label": label,
        "u_prev_final": u_prev,
    }


def summarize_result(result):
    xs = result["x"]
    us = result["u"]
    return {
        "final altitude x4": xs[-1, 3],
        "max |pitch angle x2|": np.max(np.abs(xs[:, 1])),
        "max |elevator u|": np.max(np.abs(us)) if len(us) > 0 else 0.0,
        "altitude min": np.min(xs[:, 3]),
        "altitude max": np.max(xs[:, 3]),
        "steps simulated": len(us),
    }


def make_time_axes(result):
    x = result["x"]
    u = result["u"]
    tx = np.arange(x.shape[0]) * Ts
    tu = np.arange(1, len(u) + 1) * Ts
    return tx, tu


def plot_response(result, title, subtitle="", axes=None):
    tx, tu = make_time_axes(result)
    x = result["x"]
    u = result["u"]

    created_fig = False
    if axes is None:
        fig, axes = plt.subplots(3, 1, figsize=(9.5, 8.2), sharex=True)
        created_fig = True
    else:
        fig = axes[0].figure

    axes[0].plot(tx, x[:, 3], color="#1f77b4", lw=2.2)
    axes[0].axhline(0.0, color="black", lw=0.8, ls="--")
    axes[0].set_ylabel("Altitude x4 (m)")

    axes[1].plot(tx, x[:, 1], color="#d62728", lw=2.2)
    axes[1].axhline(pitch_max, color="black", lw=0.9, ls="--", alpha=0.65)
    axes[1].axhline(-pitch_max, color="black", lw=0.9, ls="--", alpha=0.65)
    axes[1].set_ylabel("Pitch angle x2 (rad)")

    axes[2].step(tu, u, where="post", color="#2ca02c", lw=2.2)
    axes[2].axhline(u_max, color="black", lw=0.9, ls="--", alpha=0.65)
    axes[2].axhline(-u_max, color="black", lw=0.9, ls="--", alpha=0.65)
    axes[2].set_ylabel("Elevator u (rad)")
    axes[2].set_xlabel("Time (sec)")

    axes[0].set_title(title)
    if subtitle:
        axes[0].text(
            0.01,
            0.94,
            subtitle,
            transform=axes[0].transAxes,
            va="top",
            ha="left",
            bbox=dict(facecolor="white", alpha=0.92, edgecolor="#666666"),
        )

    for ax in axes:
        ax.grid(True, alpha=0.35)
        ax.set_xlim(tx[0], tx[-1] if len(tx) > 1 else 10.0)

    fig.tight_layout()
    if created_fig:
        plt.show()


display(Markdown("### Pole Check"))
display(Markdown(f"- Continuous-time open-loop poles: `{np.round(open_loop_poles_c, 4)}`"))
display(Markdown(f"- Discrete-time open-loop poles: `{np.round(open_loop_poles_d, 4)}`"))
display(Markdown(f"- Discrete-time LQR closed-loop poles: `{np.round(closed_loop_poles_d, 4)}`"))


## Reproducing the Lecture Cases

The scenarios below follow the slide sequence.

Notes:

- For the **LQR** panel, the continuous-time LQR gain is applied to the sampled plant and then saturated. This reproduces the unstable behavior shown in the lecture.
- For the **input-only MPC** panel, the controller models only the amplitude bound, but the simulated actuator still enforces the physical rate limit. This is the key reason the response fails to converge.
- For the **short-horizon** comparison, the first case uses a short horizon alone, while the second adds the infinite-horizon terminal matrix $P_\infty$.


In [ ]:
scenarios = [
    {
        "name": "LQR with saturation",
        "kind": "lqr",
        "x0": np.array([0.0, 0.0, 0.0, 10.0]),
        "note": "Continuous-time LQR gain, then input saturation only. The response is unstable, as in the lecture slide.",
    },
    {
        "name": "MPC with input bounds only",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 10.0]),
        "horizon": 10,
        "rate_bound_model": None,
        "rate_bound_actual": du_max_phys,
        "pitch_bound": None,
        "terminal_cost": P_disc,
        "note": "Controller knows only |u_k| <= 0.262, but the real actuator also enforces the rate limit. This mismatch leads to oscillation / a limit cycle.",
    },
    {
        "name": "MPC with all input constraints",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 10.0]),
        "horizon": 10,
        "rate_bound_model": du_max_phys,
        "rate_bound_actual": du_max_phys,
        "pitch_bound": None,
        "terminal_cost": P_disc,
        "note": "Amplitude and rate constraints are modeled consistently. The closed-loop response becomes well-behaved and stabilizing.",
    },
    {
        "name": "MPC without pitch constraint",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 100.0]),
        "horizon": 10,
        "rate_bound_model": du_max_phys,
        "rate_bound_actual": du_max_phys,
        "pitch_bound": None,
        "terminal_cost": P_disc,
        "note": "Large altitude step: the aircraft recovers, but the transient pitch excursion is too large for passenger comfort.",
    },
    {
        "name": "MPC with pitch constraint |x2| <= 0.349",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 100.0]),
        "horizon": 10,
        "rate_bound_model": du_max_phys,
        "rate_bound_actual": du_max_phys,
        "pitch_bound": pitch_max,
        "terminal_cost": P_disc,
        "note": "Adding the pitch-angle constraint keeps the transient near the comfort bound and still stabilizes the altitude.",
    },
    {
        "name": "Short-horizon MPC (N = 4)",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 10.0]),
        "horizon": 4,
        "rate_bound_model": du_max_tight,
        "rate_bound_actual": du_max_tight,
        "pitch_bound": None,
        "terminal_cost": Q,
        "note": "A short horizon without a strong terminal ingredient can lose stability and drives the actuator to saturation.",
    },
    {
        "name": "Short-horizon MPC (N = 4) with terminal cost",
        "kind": "mpc",
        "x0": np.array([0.0, 0.0, 0.0, 10.0]),
        "horizon": 4,
        "rate_bound_model": du_max_tight,
        "rate_bound_actual": du_max_tight,
        "pitch_bound": None,
        "terminal_cost": P_disc,
        "note": "With the terminal cost P_inf included, the short-horizon controller recovers the stabilizing behavior.",
    },
]


def run_scenario(spec, steps=40):
    if spec["kind"] == "lqr":
        result = simulate_lqr_with_saturation(
            x0=spec["x0"],
            steps=steps,
            use_continuous_gain=True,
            du_bound=None,
        )
    else:
        result = simulate_mpc(
            x0=spec["x0"],
            steps=steps,
            horizon=spec["horizon"],
            rate_bound_model=spec["rate_bound_model"],
            rate_bound_actual=spec["rate_bound_actual"],
            pitch_bound=spec["pitch_bound"],
            terminal_cost=spec["terminal_cost"],
            solver=cp.SCS,
            label=spec["name"],
        )

    result["scenario_name"] = spec["name"]
    result["note"] = spec["note"]
    return result


all_results = [run_scenario(spec) for spec in scenarios]

summary_rows = []
for spec, result in zip(scenarios, all_results):
    row = {"scenario": spec["name"]}
    row.update(summarize_result(result))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(4))

for spec, result in zip(scenarios, all_results):
    plot_response(
        result=result,
        title=spec["name"],
        subtitle=spec["note"],
    )


## Interactive Scenario Viewer

Use the widget below to rerun one scenario at a time and stress the design choices:

- change the horizon,
- switch the actuator rate limit seen by the controller,
- enable or disable the pitch-angle bound,
- toggle the terminal cost.

This is useful for seeing *why* the lecture examples differ, not just what the final plots look like.


In [ ]:
scenario_options = {spec["name"]: spec for spec in scenarios}

scenario_dd = widgets.Dropdown(
    options=list(scenario_options.keys()),
    value="MPC with all input constraints",
    description="Scenario",
    layout=widgets.Layout(width="520px"),
)

horizon_sl = widgets.IntSlider(value=10, min=3, max=20, step=1, description="Horizon", continuous_update=False)
x0_alt_sl = widgets.FloatSlider(value=10.0, min=0.0, max=120.0, step=5.0, description="x4(0)", continuous_update=False)
model_rate_dd = widgets.Dropdown(
    options=[
        ("No rate model", None),
        ("0.349 Ts", du_max_tight),
        ("0.524 Ts", du_max_phys),
    ],
    value=du_max_phys,
    description="Model rate",
    layout=widgets.Layout(width="320px"),
)
actual_rate_dd = widgets.Dropdown(
    options=[
        ("No actuator rate limit", None),
        ("0.349 Ts", du_max_tight),
        ("0.524 Ts", du_max_phys),
    ],
    value=du_max_phys,
    description="Actual rate",
    layout=widgets.Layout(width="320px"),
)
pitch_cb = widgets.Checkbox(value=False, description="Enforce |x2| <= 0.349")
term_cb = widgets.Checkbox(value=True, description="Use terminal cost P_inf")

viewer_out = widgets.Output()


def sync_controls_from_scenario(change=None):
    spec = scenario_options[scenario_dd.value]
    x0_alt_sl.value = float(spec["x0"][3])
    if spec["kind"] == "mpc":
        horizon_sl.value = int(spec["horizon"])
        model_rate_dd.value = spec["rate_bound_model"]
        actual_rate_dd.value = spec["rate_bound_actual"]
        pitch_cb.value = spec["pitch_bound"] is not None
        term_cb.value = spec["terminal_cost"] is P_disc


def update_view(_=None):
    viewer_out.clear_output(wait=True)
    spec = scenario_options[scenario_dd.value]

    with viewer_out:
        x0 = np.array([0.0, 0.0, 0.0, x0_alt_sl.value])

        if spec["kind"] == "lqr":
            result = simulate_lqr_with_saturation(
                x0=x0,
                steps=40,
                use_continuous_gain=True,
                du_bound=None,
            )
            plot_response(
                result,
                title="Interactive: LQR with saturation",
                subtitle="Continuous-time LQR gain with sampled implementation and actuator saturation.",
            )
            display(pd.DataFrame([summarize_result(result)]).round(4))
            return

        terminal_matrix = P_disc if term_cb.value else Q
        pitch_bound = pitch_max if pitch_cb.value else None

        result = simulate_mpc(
            x0=x0,
            steps=40,
            horizon=horizon_sl.value,
            rate_bound_model=model_rate_dd.value,
            rate_bound_actual=actual_rate_dd.value,
            pitch_bound=pitch_bound,
            terminal_cost=terminal_matrix,
            solver=cp.SCS,
            label="Interactive MPC",
        )

        note = (
            f"N={horizon_sl.value}, "
            f"model rate={model_rate_dd.label if hasattr(model_rate_dd, 'label') else model_rate_dd.value}, "
            f"actual rate={actual_rate_dd.label if hasattr(actual_rate_dd, 'label') else actual_rate_dd.value}"
        )
        plot_response(result, title=f"Interactive: {scenario_dd.value}", subtitle=note)
        display(pd.DataFrame([summarize_result(result)]).round(4))


scenario_dd.observe(sync_controls_from_scenario, names="value")

for w in [scenario_dd, horizon_sl, x0_alt_sl, model_rate_dd, actual_rate_dd, pitch_cb, term_cb]:
    w.observe(update_view, names="value")

sync_controls_from_scenario()

controls = widgets.VBox([
    scenario_dd,
    widgets.HBox([horizon_sl, x0_alt_sl]),
    widgets.HBox([model_rate_dd, actual_rate_dd]),
    widgets.HBox([pitch_cb, term_cb]),
])

display(widgets.VBox([controls, viewer_out]))
update_view()


# Example 2: MPC Point Feasibility and Loss of Feasibility

This code section turns the lecture's **double-integrator feasibility example** into an interactive experiment.

We study the constrained finite-horizon MPC problem

$$
x(k+1)=
\begin{bmatrix}
1 & 1 \\
0 & 1
\end{bmatrix}x(k)+
\begin{bmatrix}
0 \\
1
\end{bmatrix}u(k),
\qquad
y(k)=
\begin{bmatrix}
1 & 0
\end{bmatrix}x(k)
$$

subject to

- input constraint: $-0.5 \le u(k) \le 0.5$
- state constraint: $-5 \le x_i(k) \le 5$

The interactive plot focuses on two different notions:

- **Initial feasible set**: points where the finite-horizon optimization problem is feasible at the current time
- **Closed-loop feasible region for a chosen number of MPC steps**: points whose receding-horizon trajectory remains feasible for the selected number of closed-loop iterations

A point can therefore be feasible **now**, yet still be driven to a future state where the MPC optimization becomes infeasible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
import cvxpy as cp

from functools import lru_cache
from IPython.display import display, Markdown

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8.8, 7.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

## Model and Feasibility Setup

We use the lecture example

- default horizon: $N = 3$
- default weights: $Q = I$, $R = 10$
- terminal cost: $P = Q$
- terminal constraint: $x_N \in X$ only

The plot colors mean:

- gray: infeasible already at time $k=0$
- orange: feasible at time $k=0$, but loses feasibility within the chosen number of MPC steps
- green: remains feasible for all chosen MPC steps
- black contour: the boundary of the initial feasible set $X_N$

The black feasible boundary depends on the horizon and constraints, but not on the cost weights.

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])

x_min = np.array([-5.0, -5.0])
x_max = np.array([ 5.0,  5.0])
u_min = -0.5
u_max =  0.5

X_PLOT_MIN, X_PLOT_MAX = -5.0, 5.0
Y_PLOT_MIN, Y_PLOT_MAX = -5.0, 5.0

RESOLUTION_PRESETS = {
    "Fast": {"grid_n": 19, "traj_n": 5},
    "Balanced": {"grid_n": 25, "traj_n": 7},
    "Fine": {"grid_n": 35, "traj_n": 9},
}


def make_weights(q_scale=1.0, r=10.0):
    Q = float(q_scale) * np.eye(2)
    R = np.array([[float(r)]])
    P = Q.copy()
    return Q, R, P


@lru_cache(maxsize=60000)
def solve_mpc_cached(x1, x2, horizon, q_scale, r):
    x0 = np.array([x1, x2], dtype=float)
    Q, R, P = make_weights(q_scale, r)

    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))

    constraints = [X[:, 0] == x0]
    cost = 0

    for k in range(horizon):
        cost += cp.quad_form(X[:, k], Q) + cp.quad_form(U[:, k], R)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k],
            X[:, k] >= x_min,
            X[:, k] <= x_max,
            U[:, k] >= u_min,
            U[:, k] <= u_max,
        ]

    cost += cp.quad_form(X[:, horizon], P)
    constraints += [
        X[:, horizon] >= x_min,
        X[:, horizon] <= x_max,
    ]

    problem = cp.Problem(cp.Minimize(cost), constraints)
    problem.solve(
        solver=cp.OSQP,
        warm_start=True,
        verbose=False,
        eps_abs=1e-5,
        eps_rel=1e-5,
        max_iter=20000,
    )

    if U.value is None:
        return {"feasible": False, "u0": None, "status": problem.status}

    return {
        "feasible": True,
        "u0": float(U.value[0, 0]),
        "status": problem.status,
    }


def solve_mpc_step(x0, horizon, q_scale, r):
    x1 = float(np.round(x0[0], 8))
    x2 = float(np.round(x0[1], 8))
    return solve_mpc_cached(x1, x2, int(horizon), float(q_scale), float(r))


def simulate_closed_loop(x0, horizon, sim_steps, q_scale, r):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []

    for _ in range(sim_steps):
        sol = solve_mpc_step(x, horizon, q_scale, r)
        statuses.append(sol["status"])
        if not sol["feasible"]:
            return {
                "initial_feasible": True,
                "closed_loop_feasible": False,
                "x": np.array(xs),
                "u": np.array(us),
                "fail_state": x.copy(),
                "statuses": statuses,
            }

        u = sol["u0"]
        us.append(u)
        x = A @ x + B[:, 0] * u
        xs.append(x.copy())

        if np.any(x < x_min - 1e-8) or np.any(x > x_max + 1e-8):
            return {
                "initial_feasible": True,
                "closed_loop_feasible": False,
                "x": np.array(xs),
                "u": np.array(us),
                "fail_state": x.copy(),
                "statuses": statuses,
            }

    return {
        "initial_feasible": True,
        "closed_loop_feasible": True,
        "x": np.array(xs),
        "u": np.array(us),
        "fail_state": None,
        "statuses": statuses,
    }


def classify_point(x0, horizon, sim_steps, q_scale, r):
    first = solve_mpc_step(x0, horizon, q_scale, r)
    if not first["feasible"]:
        return 0, {
            "initial_feasible": False,
            "closed_loop_feasible": False,
            "x": np.array([x0], dtype=float),
            "u": np.array([]),
            "fail_state": np.array(x0, dtype=float),
            "statuses": [first["status"]],
        }

    sim = simulate_closed_loop(x0, horizon, sim_steps, q_scale, r)
    return (2 if sim["closed_loop_feasible"] else 1), sim


def evaluate_grid(horizon, sim_steps, q_scale, r, grid_n, progress=None):
    xs = np.linspace(X_PLOT_MIN, X_PLOT_MAX, grid_n)
    ys = np.linspace(Y_PLOT_MIN, Y_PLOT_MAX, grid_n)
    X1, X2 = np.meshgrid(xs, ys)

    closed_loop_class = np.zeros_like(X1, dtype=int)
    initial_feasible_mask = np.zeros_like(X1, dtype=bool)

    total_rows = grid_n
    for i in range(grid_n):
        for j in range(grid_n):
            x0 = np.array([X1[i, j], X2[i, j]])
            first = solve_mpc_step(x0, horizon, q_scale, r)
            if first["feasible"]:
                initial_feasible_mask[i, j] = True
                sim = simulate_closed_loop(x0, horizon, sim_steps, q_scale, r)
                closed_loop_class[i, j] = 2 if sim["closed_loop_feasible"] else 1
            else:
                closed_loop_class[i, j] = 0

        if progress is not None:
            progress("grid", i + 1, total_rows)

    return X1, X2, closed_loop_class, initial_feasible_mask


def generate_sample_trajectories(horizon, sim_steps, q_scale, r, sample_n, progress=None):
    xs = np.linspace(X_PLOT_MIN, X_PLOT_MAX, sample_n)
    ys = np.linspace(Y_PLOT_MIN, Y_PLOT_MAX, sample_n)
    trajectories = []

    total = sample_n * sample_n
    count = 0
    for x1 in xs:
        for x2 in ys:
            x0 = np.array([x1, x2], dtype=float)
            cls, sim = classify_point(x0, horizon, sim_steps, q_scale, r)
            trajectories.append({"class": cls, "sim": sim, "x0": x0})
            count += 1
            if progress is not None:
                progress("traj", count, total)

    return trajectories


def plot_feasibility_map(
    horizon,
    sim_steps,
    q_scale,
    r,
    resolution_name="Balanced",
    selected_point=None,
    progress=None,
):
    preset = RESOLUTION_PRESETS[resolution_name]
    grid_n = preset["grid_n"]
    traj_n = preset["traj_n"]

    X1, X2, closed_loop_class, initial_feasible_mask = evaluate_grid(
        horizon=horizon,
        sim_steps=sim_steps,
        q_scale=q_scale,
        r=r,
        grid_n=grid_n,
        progress=progress,
    )

    fig, ax = plt.subplots(1, 1, figsize=(8.9, 7.3))

    cmap_colors = np.array([
        [0.86, 0.86, 0.86, 1.0],
        [0.98, 0.76, 0.53, 1.0],
        [0.66, 0.86, 0.66, 1.0],
    ])
    ax.contourf(X1, X2, closed_loop_class, levels=[-0.5, 0.5, 1.5, 2.5], colors=cmap_colors)

    boundary = initial_feasible_mask.astype(float)
    ax.contour(X1, X2, boundary, levels=[0.5], colors="black", linewidths=1.8)

    trajectories = generate_sample_trajectories(
        horizon=horizon,
        sim_steps=sim_steps,
        q_scale=q_scale,
        r=r,
        sample_n=traj_n,
        progress=progress,
    )

    for item in trajectories:
        sim = item["sim"]
        xs = sim["x"]
        cls = item["class"]
        if len(xs) <= 1:
            ax.scatter(xs[0, 0], xs[0, 1], s=9, color="#444444", alpha=0.35)
            continue

        color = "#2e7d32" if cls == 2 else "#c96a00"
        ax.plot(xs[:, 0], xs[:, 1], color=color, lw=0.9, alpha=0.35)
        ax.scatter(xs[0, 0], xs[0, 1], s=10, color=color, alpha=0.35)

    if selected_point is not None:
        cls, sim = classify_point(selected_point, horizon, sim_steps, q_scale, r)
        xs = sim["x"]
        highlight_color = "#0055cc" if cls == 2 else "#aa0000" if cls == 1 else "#111111"
        ax.plot(xs[:, 0], xs[:, 1], color=highlight_color, lw=2.6, marker="o", ms=4, zorder=9)
        ax.scatter([selected_point[0]], [selected_point[1]], s=90, marker="*", color=highlight_color, edgecolor="black", zorder=10)

        if cls == 0:
            status_text = "Selected point: infeasible already at k = 0"
        elif cls == 1:
            status_text = "Selected point: feasible initially, then loses feasibility"
        else:
            status_text = f"Selected point: feasible for all {sim_steps} MPC steps"

        ax.text(
            0.02,
            0.98,
            status_text,
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(facecolor="white", alpha=0.94, edgecolor=highlight_color, linewidth=2),
        )

    ax.set_title("Point feasibility under receding-horizon MPC")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_xlim(X_PLOT_MIN, X_PLOT_MAX)
    ax.set_ylim(Y_PLOT_MIN, Y_PLOT_MAX)

    legend_handles = [
        mpatches.Patch(color=cmap_colors[0], label="Infeasible at k = 0"),
        mpatches.Patch(color=cmap_colors[1], label="Feasible now, infeasible later"),
        mpatches.Patch(color=cmap_colors[2], label=f"Feasible for all {sim_steps} MPC steps"),
        plt.Line2D([0], [0], color="black", lw=1.8, label="Initial feasible-set boundary"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", framealpha=0.95)
    ax.grid(True, alpha=0.35)
    plt.show()


sanity_seq = []
x = np.array([-4.0, 3.0])
for _ in range(3):
    sol = solve_mpc_step(x, horizon=3, q_scale=1.0, r=10.0)
    sanity_seq.append((x.copy(), sol["feasible"], sol["u0"]))
    if not sol["feasible"]:
        break
    x = A @ x + B[:, 0] * sol["u0"]

display(Markdown("### Lecture Check"))
display(Markdown(f"`x(0)=[-4,3] -> u0={sanity_seq[0][2]:.3f}`"))
display(Markdown(f"`x(1)=[-1,2.5] -> u0={sanity_seq[1][2]:.3f}`"))
display(Markdown(f"`x(2)=[1.5,2] -> feasible? {solve_mpc_step(np.array([1.5, 2.0]), 3, 1.0, 10.0)['feasible']}`"))

## Interactive Feasibility Viewer

The controls are intentionally compact:

- **Controller**: tune the horizon and quadratic weights
- **View**: choose how many closed-loop MPC steps to test and how detailed the recomputation should be
- **Selected point**: highlight one initial condition and its trajectory

A progress bar is shown while the map is being recomputed.

In [ ]:
q_scale_sl = widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05, description="Q scale", continuous_update=False)
r_sl = widgets.FloatLogSlider(value=10.0, base=10, min=-1, max=2, step=0.05, description="R", continuous_update=False)
horizon_sl = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Horizon", continuous_update=False)
sim_steps_sl = widgets.IntSlider(value=3, min=1, max=10, step=1, description="MPC steps", continuous_update=False)
resolution_dd = widgets.Dropdown(options=["Fast", "Balanced", "Fine"], value="Balanced", description="Detail")

x1_sl = widgets.FloatSlider(value=-4.0, min=-5.0, max=5.0, step=0.25, description="x1(0)", continuous_update=False)
x2_sl = widgets.FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.25, description="x2(0)", continuous_update=False)

progress_label = widgets.HTML("<b>Ready.</b>")
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description="Compute", bar_style="", layout=widgets.Layout(width="420px"))
out = widgets.Output()


def set_progress(stage, current, total):
    if total <= 0:
        return
    if stage == "grid":
        base = 0.0
        span = 75.0
        label = "Evaluating feasibility grid..."
    else:
        base = 75.0
        span = 25.0
        label = "Tracing sample trajectories..."

    progress_bar.value = int(base + span * current / total)
    progress_label.value = f"<b>{label}</b> {current}/{total}"


def finish_progress():
    progress_bar.value = 100
    progress_bar.bar_style = "success"
    progress_label.value = "<b>Done.</b>"


def start_progress():
    progress_bar.value = 0
    progress_bar.bar_style = "info"
    progress_label.value = "<b>Starting recomputation...</b>"


def update_ps11(_=None):
    start_progress()
    out.clear_output(wait=True)
    with out:
        plot_feasibility_map(
            horizon=horizon_sl.value,
            sim_steps=sim_steps_sl.value,
            q_scale=q_scale_sl.value,
            r=r_sl.value,
            resolution_name=resolution_dd.value,
            selected_point=np.array([x1_sl.value, x2_sl.value], dtype=float),
            progress=set_progress,
        )
    finish_progress()


for w in [q_scale_sl, r_sl, horizon_sl, sim_steps_sl, resolution_dd, x1_sl, x2_sl]:
    w.observe(update_ps11, names="value")


controller_box = widgets.VBox([horizon_sl, q_scale_sl, r_sl])
view_box = widgets.VBox([sim_steps_sl, resolution_dd])
point_box = widgets.VBox([x1_sl, x2_sl])

controller_card = widgets.VBox([widgets.HTML("<b>Controller</b>"), controller_box], layout=widgets.Layout(width="250px"))
view_card = widgets.VBox([widgets.HTML("<b>View</b>"), view_box], layout=widgets.Layout(width="220px"))
point_card = widgets.VBox([widgets.HTML("<b>Selected Point</b>"), point_box], layout=widgets.Layout(width="250px"))

top_row = widgets.HBox([controller_card, view_card, point_card])
progress_row = widgets.HBox([progress_bar, progress_label])

display(widgets.VBox([top_row, progress_row, out]))
update_ps11()